# LEAN Error Fixing with LangChain + Gemini

This notebook starts a LEAN server, collects diagnostics for `.lean` files, and sends files + errors to Gemini (via LangChain) to propose fixes.

In [29]:
# Install required packages
!pip install -q langchain-google-genai google-generativeai

In [30]:
import os
import json
import time
import threading
import queue
import subprocess
from pathlib import Path
from typing import Dict, List, Any, Optional
from urllib.parse import urljoin, urlunparse, quote
from langchain_google_genai import ChatGoogleGenerativeAI

In [31]:
# Configuration
MODEL_NAME = "gemini-2.5-flash"
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set in the environment")

PROJECT_ROOT = Path("extracted/CML_Lean")
LEAN_SOURCE_DIR = PROJECT_ROOT / "CML_Lean"

# Lean server command (stdio LSP)
LEAN_SERVER_CMD = ["lake", "env", "lean", "--server"]

In [34]:
# Preflight: verify lake + lean toolchain
def run_cmd(cmd: List[str], cwd: Path) -> str:
    result = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{result.stderr}")
    return result.stdout.strip()

print(run_cmd(["lake", "--version"], PROJECT_ROOT))
print(run_cmd(["lake", "env", "lean", "--version"], LEAN_SOURCE_DIR))

Lake version 5.0.0-src+db93fe1 (Lean version 4.27.0)
Lean (version 4.27.0, x86_64-w64-windows-gnu, commit db93fe1608548721853390a10cd40580fe7d22ae, Release)
Lean (version 4.27.0, x86_64-w64-windows-gnu, commit db93fe1608548721853390a10cd40580fe7d22ae, Release)


In [35]:
# Minimal LSP (JSON-RPC) client to receive Lean diagnostics
class LspClient:
    def __init__(self, cmd: List[str], cwd: Path, *, debug_io: bool = False):
        self.proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False,
            bufsize=0
        )
        self._id = 0
        self._responses: Dict[int, Any] = {}
        self._notifications: queue.Queue = queue.Queue()
        self._stderr_lines: List[str] = []
        self._debug_io = debug_io
        self._io_errors: List[str] = []
        self._reader_thread = threading.Thread(target=self._read_loop, daemon=True)
        self._stderr_thread = threading.Thread(target=self._read_stderr, daemon=True)
        self._reader_thread.start()
        self._stderr_thread.start()

    def _read_loop(self):
        buf = b""
        while True:
            chunk = self.proc.stdout.read(4096)
            if not chunk:
                return
            buf += chunk
            while True:
                header_end = buf.find(b"\r\n\r\n")
                if header_end == -1:
                    break
                header_bytes = buf[:header_end]
                rest = buf[header_end + 4 :]
                header_text = header_bytes.decode("utf-8", errors="replace")
                content_length = None
                for line in header_text.split("\r\n"):
                    if line.lower().startswith("content-length:"):
                        try:
                            content_length = int(line.split(":", 1)[1].strip())
                        except Exception:
                            content_length = None
                        break
                if content_length is None:
                    buf = buf[1:]
                    continue
                if len(rest) < content_length:
                    break
                body = rest[:content_length]
                buf = rest[content_length:]
                try:
                    msg = json.loads(body.decode("utf-8"))
                except Exception as e:
                    err = f"JSON decode failed: {e}. Body len={len(body)}"
                    self._io_errors.append(err)
                    if self._debug_io:
                        try:
                            Path("lean_lsp_bad_message.bin").write_bytes(body)
                        except Exception:
                            pass
                    continue
                if "id" in msg:
                    self._responses[msg["id"]] = msg
                else:
                    self._notifications.put(msg)

    def _read_stderr(self):
        while True:
            line = self.proc.stderr.readline()
            if not line:
                break
            self._stderr_lines.append(line.decode("utf-8", errors="ignore").rstrip())
            if len(self._stderr_lines) > 200:
                self._stderr_lines = self._stderr_lines[-200:]

    def _send(self, payload: Dict[str, Any]):
        data = json.dumps(payload).encode("utf-8")
        content = b"Content-Length: " + str(len(data)).encode("ascii") + b"\r\n\r\n" + data
        self.proc.stdin.write(content)
        self.proc.stdin.flush()

    def request(self, method: str, params: Dict[str, Any], timeout: float = 30.0) -> Dict[str, Any]:
        self._id += 1
        req_id = self._id
        self._send({"jsonrpc": "2.0", "id": req_id, "method": method, "params": params})
        start = time.time()
        while time.time() - start < timeout:
            if req_id in self._responses:
                return self._responses.pop(req_id)
            if self.proc.poll() is not None:
                stderr_tail = "\n".join(self._stderr_lines[-20:])
                io_tail = "\n".join(self._io_errors[-5:])
                raise RuntimeError(
                    f"Lean server exited early (code {self.proc.returncode}).\n"
                    f"IO errors:\n{io_tail}\n\nStderr:\n{stderr_tail}"
                )
            time.sleep(0.05)
        stderr_tail = "\n".join(self._stderr_lines[-20:])
        io_tail = "\n".join(self._io_errors[-5:])
        raise TimeoutError(f"No response for {method}.\nIO errors:\n{io_tail}\n\nStderr:\n{stderr_tail}")

    def notify(self, method: str, params: Dict[str, Any]):
        self._send({"jsonrpc": "2.0", "method": method, "params": params})

    def next_notification(self, timeout: float = 5.0) -> Optional[Dict[str, Any]]:
        try:
            return self._notifications.get(timeout=timeout)
        except queue.Empty:
            return None

    def close(self):
        try:
            self.proc.terminate()
        except Exception:
            pass

def path_to_uri(path: Path) -> str:
    return "file:///" + quote(str(path.resolve()).replace("\\", "/"))

def _diagnostics_signature(diags: List[Dict[str, Any]]) -> str:
    parts = []
    for d in diags:
        rng = d.get("range", {})
        start = rng.get("start", {})
        msg = d.get("message", "")
        sev = d.get("severity", None)
        parts.append(f"{start.get('line')}:{start.get('character')}|{sev}|{msg}")
    return "\n".join(parts)

def start_lean_server(project_root: Path, *, debug_io: bool = False) -> LspClient:
    client = LspClient(LEAN_SERVER_CMD, project_root, debug_io=debug_io)
    root_uri = path_to_uri(project_root)
    client.request("initialize", {
        "processId": None,
        "rootUri": root_uri,
        "workspaceFolders": [
            {"uri": root_uri, "name": project_root.name}
        ],
        "capabilities": {
            "workspace": {"workspaceFolders": True}
        },
        "initializationOptions": {
            "editDelay": 10,
            "hasWidgets": False,
            "documentFormatting": False,
            "maxNumberOfProblems": 10000
        }
    }, timeout=60.0)
    client.notify("initialized", {})
    return client

def collect_diagnostics(
    client: LspClient,
    file_paths: List[Path],
    per_file_timeout: float = 90.0,
    settle_seconds: float = 3.0,
    min_wait_seconds: float = 2.0,
 ) -> Dict[str, Optional[List[Dict[str, Any]]]]:
    """Collect diagnostics by waiting for publishDiagnostics to arrive and settle.

    Notes:
    - Lean can publish diagnostics multiple times per file as it elaborates imports.
    - If a file never receives publishDiagnostics within timeout, result is None.
    """
    diagnostics: Dict[str, Optional[List[Dict[str, Any]]]] = {}
    for path in file_paths:
        uri = path_to_uri(path)
        text = path.read_text(encoding="utf-8")
        client.notify("textDocument/didOpen", {
            "textDocument": {
                "uri": uri,
                "languageId": "lean",
                "version": 1,
                "text": text
            }
        })
        # didSave tends to trigger the same work VS Code does
        client.notify("textDocument/didSave", {
            "textDocument": {"uri": uri},
            "text": text
        })
        last_diags: Optional[List[Dict[str, Any]]] = None
        last_sig: Optional[str] = None
        last_change_time: Optional[float] = None
        start = time.time()
        while time.time() - start < per_file_timeout:
            msg = client.next_notification(timeout=1.0)
            now = time.time()
            if msg is None:
                # Ensure we wait at least a bit, even if idle
                if (now - start) < min_wait_seconds:
                    continue
            else:
                method = msg.get("method")
                if method == "textDocument/publishDiagnostics":
                    params = msg.get("params", {})
                    if params.get("uri") == uri:
                        current = params.get("diagnostics", [])
                        sig = _diagnostics_signature(current)
                        if sig != last_sig:
                            last_sig = sig
                            last_diags = current
                            last_change_time = now
                # Some servers send background progress; we just ignore it here but it keeps loop alive
            if last_change_time is not None and (now - last_change_time) >= settle_seconds and (now - start) >= min_wait_seconds:
                break
        diagnostics[str(path)] = last_diags
    return diagnostics

In [36]:
# Collect diagnostics from all .lean files (debug_io=True writes bad frames to lean_lsp_bad_message.bin)
lean_files = sorted(LEAN_SOURCE_DIR.glob("*.lean"))
if not lean_files:
    raise FileNotFoundError(f"No .lean files found in {LEAN_SOURCE_DIR}")

client = start_lean_server(PROJECT_ROOT, debug_io=True)
diagnostics = collect_diagnostics(client, lean_files, per_file_timeout=90.0, settle_seconds=3.0)
client.close()

# Show a quick summary
for path, diags in diagnostics.items():
    if diags is None:
        print(f"{path}: no diagnostics received")
    else:
        print(f"{path}: {len(diags)} diagnostic(s)")

RuntimeError: Lean server exited early (code 1).
IO errors:


Stderr:
error: compiled configuration is invalid; run with '-R' to reconfigure

In [26]:
# Gemini (LangChain) fixer
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    api_key=API_KEY,
    temperature=0.0,
    max_output_tokens=8192,
 )

def build_prompt(file_path: Path, file_text: str, file_diags: List[Dict[str, Any]]) -> str:
    return (
        "You are an expert Lean 4 assistant.\n"
        "Fix the Lean file based on the diagnostics.\n"
        "Return ONLY valid JSON: {\"fixed\": true|false, \"content\": \"<full file text>\"}.\n"
        "Do not wrap the JSON in Markdown fences.\n\n"
        f"FILE PATH: {file_path}\n\n"
        "DIAGNOSTICS (JSON):\n"
        f"{json.dumps(file_diags, ensure_ascii=False, indent=2)}\n\n"
        "FILE CONTENT:\n"
        f"{file_text}\n"
    )

def _extract_json_object(text: str) -> str:
    """Extract the first JSON object from model output.

    Handles common cases like ```json ... ``` fences or extra prose around JSON.
    """
    s = text.strip()
    if s.startswith("```"):
        lines = s.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        s = "\n".join(lines).strip()
    start = s.find("{")
    if start == -1:
        return s
    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start : i + 1]
    return s[start:]

def apply_fixes(diagnostics: Dict[str, Optional[List[Dict[str, Any]]]], dry_run: bool = False) -> Dict[str, str]:
    results: Dict[str, str] = {}
    for path_str, diags in diagnostics.items():
        if diags is None or not diags:
            continue
        path = Path(path_str)
        file_text = path.read_text(encoding="utf-8")
        prompt = build_prompt(path, file_text, diags)
        response = llm.invoke(prompt)
        raw = response.content if isinstance(response.content, str) else str(response.content)
        try:
            json_text = _extract_json_object(raw)
            payload = json.loads(json_text)
            if not isinstance(payload, dict) or "content" not in payload:
                raise ValueError("Invalid response shape")
            fixed_text = payload["content"]
        except Exception as e:
            raise ValueError(f"Failed to parse Gemini response for {path}: {e}\nRaw response:\n{raw}")
        if not dry_run:
            backup = path.with_suffix(path.suffix + ".bak")
            backup.write_text(file_text, encoding="utf-8")
            path.write_text(fixed_text, encoding="utf-8")
        results[str(path)] = "fixed"
    return results

In [27]:
# Test with location.lean first
location_file = LEAN_SOURCE_DIR / "location.lean"
if not location_file.exists():
    raise FileNotFoundError(f"Missing: {location_file}")

client = start_lean_server(PROJECT_ROOT)
location_diagnostics = collect_diagnostics(client, [location_file])
client.close()

print("location.lean diagnostics:")
print(json.dumps(location_diagnostics[str(location_file)], indent=2))

# Apply Gemini fix only to location.lean (set dry_run=True to preview)
location_results = apply_fixes(location_diagnostics, dry_run=False)
print(json.dumps(location_results, indent=2))

location.lean diagnostics:
[
  {
    "fullRange": {
      "end": {
        "character": 31,
        "line": 139
      },
      "start": {
        "character": 29,
        "line": 139
      }
    },
    "message": "unexpected token ':='; expected command",
    "range": {
      "end": {
        "character": 31,
        "line": 139
      },
      "start": {
        "character": 29,
        "line": 139
      }
    },
    "severity": 1,
    "source": "Lean 4"
  },
  {
    "fullRange": {
      "end": {
        "character": 23,
        "line": 147
      },
      "start": {
        "character": 21,
        "line": 147
      }
    },
    "message": "unexpected token ':='; expected command",
    "range": {
      "end": {
        "character": 23,
        "line": 147
      },
      "start": {
        "character": 21,
        "line": 147
      }
    },
    "severity": 1,
    "source": "Lean 4"
  },
  {
    "fullRange": {
      "end": {
        "character": 20,
        "line": 102
      },
      "st

In [28]:
# Apply fixes (set dry_run=True to preview without writing)
results = apply_fixes(diagnostics, dry_run=False)
print(json.dumps(results, indent=2))

ValueError: Failed to parse Gemini response for extracted\CML_Lean\CML_Lean\ast.lean: Invalid control character at: line 1 column 80 (char 79)
Raw response:
{"fixed": true, "content": "-- Auto-generated LEAN 4 file from HOL4 translation
-- Theory: ast
-- Generated using Gemini API

-- The following imports are commented out because the diagnostic indicates
-- that these modules failed to build, making their contents unavailable.
-- Placeholder definitions are provided below to allow this file to compile
-- in isolation, addressing the dependency failure reported in the diagnostics.
-- import CML_Lean.namespace
-- import CML_Lean.location

-- Placeholder definitions for types expected from CML_Lean.namespace
-- Inferred from usage in ast.lean and common patterns for qualified identifiers.
structure cml_id (NameType : Type) (ModuleType : Type) where
  name : NameType
  module : Option ModuleType

-- Placeholder definitions for types expected from CML_Lean.location
-- Inferred from usage 

In [ ]:
# After location.lean passes, apply to all .lean files
client = start_lean_server(PROJECT_ROOT)
all_diagnostics = collect_diagnostics(client, lean_files)
client.close()

all_results = apply_fixes(all_diagnostics, dry_run=False)
print(json.dumps(all_results, indent=2))

{}


In [ ]:
# Debug: show diagnostics for a specific file (match VS Code as closely as possible)
target = LEAN_SOURCE_DIR / "location.lean"
client = start_lean_server(PROJECT_ROOT, debug_io=True)
diag_map = collect_diagnostics(client, [target], per_file_timeout=180.0, settle_seconds=3.0, min_wait_seconds=3.0)
client.close()

diags = diag_map[str(target)]
print(f"Diagnostics received for {target}: {0 if (diags is None) else len(diags)}")
print(json.dumps(diags, indent=2, ensure_ascii=False))

Diagnostics received for extracted\CML_Lean\CML_Lean\location.lean: 0
[]


In [ ]:
# CLI cross-check (if LSP still shows 0 diagnostics)
# This shows whether Lean/Lake actually sees errors in the project.
import subprocess
print(subprocess.run(["lake", "build"], cwd=str(PROJECT_ROOT), capture_output=True, text=True).stdout)
print(subprocess.run(["lake", "build"], cwd=str(PROJECT_ROOT), capture_output=True, text=True).stderr)